In [ ]:
#AI was used to help generate this code. Code was tested and verified.
#Model used to generate code: qwen-coder:30b

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

%matplotlib inline

In [30]:
def load_transactions(file_path):

    try:
        #Use pandas to read csv file in absolute path. See last block. 
        df = pd.read_csv(file_path)
        
        #Checking all columns are in CSV file
        required_columns = ['Type', 'Account', 'Item', 'Category', 'Amount', 'Date']
        if not all(col in df.columns for col in required_columns):
            raise ValueError(f"CSV must contain columns: {required_columns}")
        
        #Convert Date column to datetime
        df['Date'] = pd.to_datetime(df['Date'])
        
        #Sort by date. Reset index and get rid of old index
        df = df.sort_values('Date').reset_index(drop=True)
        
        return df
    except Exception as e:
        print(f"Error loading CSV file: {e}")
        raise  

In [31]:
def create_mirror_transfers(df):
    #Create mirror transfer transactions by swapping Account and Item field values
    # Filter for transfer transactions
    transfer_df = df[df['Type'] == 'Transfer'].copy()
    
    #Swap the values in Account and Item columns.
    temp = transfer_df['Account'].copy()
    transfer_df['Account'] = transfer_df['Item']
    transfer_df['Item'] = temp
    
    #Make the amounts negative
    transfer_df['Amount'] = -transfer_df['Amount']

    #Reset index
    transfer_df = transfer_df.reset_index(drop=True)
    
    return transfer_df

In [32]:
def calculate_account_balances(df, transfers_df=None):
    #Calculate running balance for each account
    #Create a copy to avoid modifying original data
    df_copy = df.copy()
    
    #If transfers_df is provided, merge it with the original df
    if transfers_df is not None:
        #Combine original transactions with mirror transfers
        df_copy = pd.concat([df_copy, transfers_df], ignore_index=True)
        #Sort by date again after merging
        df_copy = df_copy.sort_values('Date').reset_index(drop=True)
    
    #Create empty dictionary to track balances
    account_balances = {}
    
    #Process each transaction
    for idx, row in df_copy.iterrows():
        transaction_type = row['Type']
        account = row['Account']
        amount = row['Amount']
        
        if account not in account_balances:
            account_balances[account] = 0

        if transaction_type == "Withdrawal":
            account_balances[account] -= amount
        elif transaction_type == "Deposit":
            account_balances[account] += amount
        elif transaction_type == "Credit Payment":
            account_balances['Account 3'] -= amount
            account_balances['Credit Charge'] += amount
        elif transaction_type == "Transfer":
            account_balances[account] -= amount
        df_copy.loc[idx, 'Balance'] = account_balances[account]
    
    return df_copy


In [33]:
def print_ending_balances(df_with_balances):
    #Get and print the ending balance for each account"
    #Get the ending balance for each account 
    ending_balances = df_with_balances.groupby('Account')['Balance'].last()
    
    print("\n--- Ending Balances ---")
    for account, balance in ending_balances.items():
        #Don't Show Credit Payment
        if account != 'Credit Payment':
            print(f"{account}: ${balance:.2f}")
    
    #Print total balance across all accounts (excluding Credit Payment)
    total_balance = ending_balances.sum()
    print(f"\nTotal Balance: ${total_balance:.2f}")

In [ ]:
def plot_account_balances(df):
    #Ensure Date column is datetime
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    
    #Filter out Credit Payment transactions for plotting
    df_plot = df[df['Type'] != 'Credit Payment']
    
    #Create a pivot table for plotting
    pivot_df = df_plot.pivot_table(index='Date', columns='Account', values='Balance', aggfunc='last')
    
    #Create the plot
    plt.figure(figsize=(12, 8))
    
    #Plot each account
    for account in pivot_df.columns:
        plt.scatter(pivot_df.index, pivot_df[account], marker='o', label=account)
    
    plt.title('Account Balances Over Time')
    plt.xlabel('Date')
    plt.ylabel('Balance')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    # Show the plot
    plt.show()


In [ ]:
def plot_account_balances_eom(df):
    #Ensure Date column is datetime
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    
    #Filter out Credit Payment transactions for plotting
    df_plot = df[df['Type'] != 'Credit Payment']
    
    #Create a pivot table for plotting
    pivot_df = df_plot.pivot_table(index='Date', columns='Account', values='Balance', aggfunc='last')
    
    # Group by month and get the last balance of each month
    monthly_balances = pivot_df.groupby(pivot_df.index.to_period('M')).last()

    #Create the plot
    plt.figure(figsize=(12, 8))
    
    #Plot each account
    for account in monthly_balances.columns:
        plt.plot(monthly_balances.index.to_timestamp(), monthly_balances[account], marker='o', label=account)
    
    plt.title('Account Balances Over Time (By Month)')
    plt.ylabel('Balance')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    # Show the plot
    plt.show()

plot_account_balances_eom(df_with_balances)


In [ ]:
def plot_expenses_vs_income_by_month(df):
    # Ensure Date column is datetime
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    
    # Filter out Credit Payment transactions
    df_filtered = df[df['Type'] != 'Credit Payment']
    
    # Create monthly groups
    df_filtered['Month'] = df_filtered['Date'].dt.to_period('M')
    
    # Calculate expenses (sum of Withdrawal amounts)
    expenses = df_filtered[df_filtered['Type'] == 'Withdrawal'].groupby('Month')['Amount'].sum()
    
    # Calculate income (sum of Deposit amounts with "Paycheck" in Item)
    income = df_filtered[(df_filtered['Type'] == 'Deposit') & (df_filtered['Item'].str.contains('Paycheck', na=False))].groupby('Month')['Amount'].sum()
    
    # Create the plot
    plt.figure(figsize=(12, 8))
    
    # Plot expenses line
    plt.plot(expenses.index.to_timestamp(), expenses.values, marker='o', label='Expenses', color='red')
    
    # Plot income line
    plt.plot(income.index.to_timestamp(), income.values, marker='s', label='Income', color='green')
    
    plt.title('Expenses vs Income by Month')
    plt.ylabel('Amount ($)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    # Show the plot
    plt.show()

plot_expenses_vs_income_by_month(df)

In [ ]:
def plot_monthly_expense_breakdown_filtered_date(df):
    # Ensure Date column is datetime
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    
    # Filter out Credit Payment transactions
    df_filtered = df[df['Type'] != 'Credit Payment']
    
    # Filter only Withdrawal transactions
    withdrawals = df_filtered[df_filtered['Type'] == 'Withdrawal']
    
    # Filter for date range: June 2024 to February 2026
    start_date = pd.to_datetime('2024-06-01')
    end_date = pd.to_datetime('2026-02-28')
    withdrawals = withdrawals[(withdrawals['Date'] >= start_date) & (withdrawals['Date'] <= end_date)]
    
    # Filter out Travel and Gifts categories
    withdrawals = withdrawals[~withdrawals['Category'].isin(['Travel', 'Gifts','Rent'])]
    
    # Create monthly groups
    withdrawals['Month'] = withdrawals['Date'].dt.to_period('M')
    
    # Get all unique months in the date range
    all_months = pd.period_range(start=start_date, end=end_date, freq='M')
    
    # Group by month and category, sum the amounts
    monthly_category_sums = withdrawals.groupby(['Month', 'Category'])['Amount'].sum().reset_index()
    
    # Create a complete grid of all months and categories
    all_combinations = pd.MultiIndex.from_product([all_months, withdrawals['Category'].unique()], names=['Month', 'Category']).to_frame(index=False)
    
    # Merge to include months with no transactions
    complete_data = all_combinations.merge(monthly_category_sums, on=['Month', 'Category'], how='left')
    
    # Fill NaN values with 0
    complete_data['Amount'] = complete_data['Amount'].fillna(0)
    
    # Calculate percentage of each category per month
    monthly_totals = complete_data.groupby('Month')['Amount'].sum()
    complete_data['Percentage'] = complete_data.groupby('Month')['Amount'].transform(lambda x: x / x.sum() * 100 if x.sum() != 0 else 0)
    
    # Calculate average amount and percentage for each category across all months
    avg_expenses = complete_data.groupby('Category')['Amount'].mean()
    
    # Create pie chart with average amounts
    plt.figure(figsize=(10, 8))
    
    # Create pie chart
    plt.pie(avg_expenses.values, labels=avg_expenses.index, autopct='%1.1f%%')
    plt.title('Average Monthly Expenses by Category (June 2024 - February 2026)\n(Excluding Rent, Travel, and Gifts)')
    plt.axis('equal')
    
    plt.show()
    
    return avg_expenses

plot_monthly_expense_breakdown_filtered_date(df)

In [ ]:
def get_category_sums_and_averages(df):
    # Ensure Date column is datetime
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    
    # Filter out Credit Payment transactions
    df_filtered = df[df['Type'] != 'Credit Payment']
    
    # Filter only Withdrawal transactions
    withdrawals = df_filtered[df_filtered['Type'] == 'Withdrawal']
    
    # Filter for date range: June 2024 to February 2026
    start_date = pd.to_datetime('2024-06-01')
    end_date = pd.to_datetime('2026-02-28')
    withdrawals = withdrawals[(withdrawals['Date'] >= start_date) & (withdrawals['Date'] <= end_date)]
    
    # Create monthly groups
    withdrawals['Month'] = withdrawals['Date'].dt.to_period('M')
    
    # Get all unique months in the date range
    all_months = pd.period_range(start=start_date, end=end_date, freq='M')
    
    # Group by month and category, sum amounts
    monthly_category_sums = withdrawals.groupby(['Month', 'Category'])['Amount'].sum().reset_index()
    
    # Create a complete grid of all months and all categories
    all_combinations = pd.MultiIndex.from_product([all_months, withdrawals['Category'].unique()], names=['Month', 'Category']).to_frame(index=False)
    
    # Merge to include months with no transactions (will have NaN amounts)
    complete_data = all_combinations.merge(monthly_category_sums, on=['Month', 'Category'], how='left')
    
    # Fill NaN values with 0 (months with no spending in that category)
    complete_data['Amount'] = complete_data['Amount'].fillna(0)
    
    # Calculate total sum for each category
    total_sums = complete_data.groupby('Category')['Amount'].sum()
    
    # Calculate average monthly sum for each category (includes months with $0)
    average_sums = complete_data.groupby('Category')['Amount'].mean()
    
    return total_sums, average_sums

def display_category_sums_and_averages(df):
    """
    Displays the total and average monthly sums for each category in a formatted table.
    """
    total_sums, average_sums = get_category_sums_and_averages(df)
    
    # Create a DataFrame for better display
    result_df = pd.DataFrame({
        'Total Amount': total_sums,
        'Average Monthly': average_sums
    })
    
    # Round to 2 decimal places
    result_df = result_df.round(2)
    
    print("Category Spending Summary (June 2024 - February 2026)")
    print("=" * 50)
    print(result_df)
    print("=" * 50)
    print(f"Total spending across all categories: ${total_sums.sum():,.2f}")
    
    return result_df

# Example usage:
# totals, averages = get_category_sums_and_averages(df)
# display_category_sums_and_averages(df)

display_category_sums_and_averages(df)

In [ ]:
file_path = r"X:\Absolute\\Path\transactions_matplotlib.csv"  # Replace with actual path

# Load transactions
df = load_transactions(file_path)
transfers_df = create_mirror_transfers(df)

# Calculate balances
df_with_balances = calculate_account_balances(df, transfers_df)

# Print ending balances
print_ending_balances(df_with_balances)

# Plot account balances
plot_account_balances_eom(df_with_balances)
